# FUS protein–RNA molecular dynamics analysis

Post-processing and structural analysis of GROMACS molecular dynamics simulations
of the FUS (Fused in Sarcoma) protein–RNA system. Covers the full pipeline from
raw trajectory handling through to contact-frequency heatmaps, using both GROMACS
command-line tools and MDAnalysis.

The system studied is the FUS zinc-finger (ZnF) and RRM domains in complex with
RNA (PDB: 6G99, 6GBM, 6SNJ).

> **External dependencies:** MDAnalysis is required for sections §3–§5.
> The alchemical free-energy workflow built on top of these simulations uses
> [SMArt](https://github.com/drazen-petrov/SMArt) (Dražen Petrov / collaborators) —
> see project 03 for the EDS topology and BAR analysis sections.

**Contents**

1. [GROMACS trajectory post-processing](#sec1)
2. [Simulation observable plotting](#sec2)
3. [Protein–RNA distance map (MDAnalysis)](#sec3)
4. [Contact counting over trajectory](#sec4)
5. [Residue-level contact frequency heatmap](#sec5)

> Consolidated from separate working scripts and notebooks. Each section keeps its own
> imports and reads independently. Cell outputs were cleared; input data files are
> referenced by generic names (not included).


<a id="sec1"></a>
## 1. GROMACS trajectory post-processing (Bash)

A complete GROMACS post-processing pipeline for the FUS ZnF–RNA complex system: trajectory centering and fitting (`trjconv`), energy extraction across equilibration stages (minimisation → NVT → NPT → production), radius of gyration, and RMSD for the complex, protein, and RNA components.

<sub>Source script: <code>analysis_script_ZnF_complex.sh</code></sub>

In [ ]:
%%bash
# FUS ZnF–RNA GROMACS post-processing pipeline
# Paths and index groups are system-specific; adapt for your system.

cd <simulation_dir>
echo "$pwd"

gmx trjconv -s 6g99_ZnF_UGGUG_md.tpr -f 6g99_ZnF_UGGUG_md.xtc -n 6g99_index.ndx -o 6g99_ZnF_UGGUG_md_stripped.xtc -pbc nojump -center<<AAA
20
20
AAA
echo 20 | gmx trjconv -s 6g99_ZnF_UGGUG_md.tpr -f 6g99_ZnF_UGGUG_md.xtc -n 6g99_index.ndx -o 6g99_ZnF_UGGUG_md_stripped.gro -dump 0
gmx trjconv -s 6g99_ZnF_UGGUG_md.tpr -f 6g99_ZnF_UGGUG_md_stripped.xtc -n 6g99_index.ndx -o 6g99_ZnF_UGGUG_md_strip_fit.xtc -fit rot+trans<<AAA
20
20
AAA

gmx energy -f 6g99_ZnF_UGGUG_min.edr -o 6g99_complex_min_energy.xvg<<AAA
10
0
AAA

gmx energy -f 6g99_ZnF_UGGUG_nvt1.edr -o 6g99_complex_nvt1_temp.xvg<<BBB
14
0
BBB

gmx energy -f 6g99_ZnF_UGGUG_nvt2.edr -o 6g99_complex_nvt2_temp.xvg<<BBB
14
0
BBB

gmx energy -f 6g99_ZnF_UGGUG_nvt3.edr -o 6g99_complex_nvt3_temp.xvg<<BBB
14
0
BBB

gmx energy -f 6g99_ZnF_UGGUG_nvt4.edr -o 6g99_complex_nvt4_temp.xvg<<BBB
14
0
BBB

gmx energy -f 6g99_ZnF_UGGUG_npt.edr -o 6g99_complex_npt_pres.xvg<<CCC
15
0
CCC

gmx energy -f 6g99_ZnF_UGGUG_npt.edr -o 6g99_complex_npt_dens.xvg<<CCC
21
0
CCC

gmx energy -f 6g99_ZnF_UGGUG_md.edr -o 6g99_complex_md_tot-energy.xvg<<DDD
12
0
DDD

gmx energy -f 6g99_ZnF_UGGUG_md.edr -o 6g99_complex_md_temp.xvg<<DDD
14
0
DDD

gmx energy -f 6g99_ZnF_UGGUG_md.edr -o 6g99_complex_md_pres.xvg<<DDD
15
0
DDD

echo 20 | gmx gyrate -s 6g99_ZnF_UGGUG_md.tpr -f 6g99_ZnF_UGGUG_md_stripped.xtc -n 6g99_index.ndx -o 6g99_complex_md_gyr.xvg

gmx rms -s 6g99_ZnF_UGGUG_md.tpr -f 6g99_ZnF_UGGUG_md_stripped.xtc -n 6g99_index.ndx -o 6g99_complex_md_RMSD_complex.xvg<<EEE
20
20
EEE


gmx rms -s 6g99_ZnF_UGGUG_md.tpr -f 6g99_ZnF_UGGUG_md_stripped.xtc -n 6g99_index.ndx -o 6g99_complex_md_RMSD_prot.xvg<<EEE
4
4
EEE

gmx energy -f 6g99_ZnF_UGGUG_md.edr -o 6g99_complex_md_pres.xvg<<DDD
15
0
DDD
mx rms -s 6g99_ZnF_UGGUG_md.tpr -f 6g99_ZnF_UGGUG_md_stripped.xtc -n 6g99_index.ndx -o 6g99_complex_md_RMSD_RNA.xvg<<EEE
1
1
EEE





<a id="sec2"></a>
## 2. Simulation observable plotting

Loads GROMACS `.xvg` output files (RMSD, RMSF, radius of gyration, energy) and plots them. Each cell handles one observable for the FUS ZnF–UGGUG system.

<sub>Source script: <code>Plotting.ipynb</code></sub>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load the .xvg file
# Use 'skiprows' to skip the header lines (usually starting with '@' or '#' in .xvg files)
data = np.loadtxt('6g99_ZnF_UGGUG_complex_gyr.xvg', comments=('@', '#'))

# Extract columns
x = data[:, 0]  # First column (typically time or step)
y = data[:, 1]  # Second column (typically the value you're plotting)

# Plot the data
plt.plot(x, y)
plt.xlabel('Time')  # Label for x-axis
plt.ylabel('RGyr')            # Label for y-axis
plt.title('Complex R gyr')  # Title of the plot

# Show the plot
plt.savefig('6g99_ZnF_UGGUG_complex_gyr.png', dpi=100)
plt.show()

In [ ]:

# Load the .xvg file
# Use 'skiprows' to skip the header lines (usually starting with '@' or '#' in .xvg files)
data = np.loadtxt('6g99_ZnF_UGGUG_complex_rms.xvg', comments=('@', '#'))

# Extract columns
x = data[:, 0]  # First column (typically time or step)
y = data[:, 1]  # Second column (typically the value you're plotting)

# Plot the data
plt.plot(x, y)
plt.xlabel('Time')  # Label for x-axis
plt.ylabel('RMSD')            # Label for y-axis
plt.title('RMSD complex')  # Title of the plot

# Show the plot
plt.savefig('6g99_ZnF_UGGUG_complex_rms.png', dpi=100)
plt.show()

In [ ]:

# Load the .xvg file
# Use 'skiprows' to skip the header lines (usually starting with '@' or '#' in .xvg files)
data = np.loadtxt('6g99_ZnF_UGGUG_md_cont_num.xvg', comments=('@', '#'))

# Extract columns
x = data[:, 0]  # First column (typically time or step)
y = data[:, 1]  # Second column (typically the value you're plotting)

# Plot the data
plt.plot(x, y)
plt.xlabel('Time')  # Label for x-axis
plt.ylabel('Contact number')            # Label for y-axis
plt.title('number of contacts 0.3 nm')  # Title of the plot

# Show the plot
plt.savefig('6g99_ZnF_UGGUG_md_cont_num.png', dpi=100)
plt.show()

In [ ]:
%cd <simulation_dir>
%pwd
%ls

In [ ]:

# Load the .xvg file
# Use 'skiprows' to skip the header lines (usually starting with '@' or '#' in .xvg files)
data = np.loadtxt('6g99_ZnF_UGGUG_protein_rms.xvg', comments=('@', '#'))

# Extract columns
x = data[:, 0]  # First column (typically time or step)
y = data[:, 1]  # Second column (typically the value you're plotting)

# Plot the data
plt.plot(x, y)
plt.xlabel('Time')  # Label for x-axis
plt.ylabel('RMSD')            # Label for y-axis
plt.title('RMSD protein')  # Title of the plot

# Show the plot
plt.savefig('6g99_ZnF_UGGUG_protein_rms.png', dpi=100)
plt.show()

In [ ]:

# Load the .xvg file
# Use 'skiprows' to skip the header lines (usually starting with '@' or '#' in .xvg files)
data = np.loadtxt('6g99_ZnF_UGGUG_RNA_rms.xvg', comments=('@', '#'))

# Extract columns
x = data[:, 0]  # First column (typically time or step)
y = data[:, 1]  # Second column (typically the value you're plotting)

# Plot the data
plt.plot(x, y)
plt.xlabel('Time')  # Label for x-axis
plt.ylabel('RMSD')            # Label for y-axis
plt.title('RMSD RNA')  # Title of the plot

# Show the plot
plt.savefig('6g99_ZnF_UGGUG_RNA_rms.png', dpi=100)
plt.show()

In [ ]:

# Load the .xvg file
# Use 'skiprows' to skip the header lines (usually starting with '@' or '#' in .xvg files)
data = np.loadtxt('6g99_ZnF_UGGUG_RNA_rmsf.xvg', comments=('@', '#'))

# Extract columns
x = data[:, 0]  # First column (typically time or step)
y = data[:, 1]  # Second column (typically the value you're plotting)

# Plot the data
plt.plot(x, y)
plt.xlabel('Residue')  # Label for x-axis
plt.ylabel('RMSF')            # Label for y-axis
plt.title('RMSF RNA')  # Title of the plot

# Show the plot
plt.savefig('6g99_ZnF_UGGUG_RNA_rmsf.png', dpi=100)
plt.show()

In [ ]:

# Load the .xvg file
# Use 'skiprows' to skip the header lines (usually starting with '@' or '#' in .xvg files)
data = np.loadtxt('6g99_ZnF_UGGUG_protein_rmsf.xvg', comments=('@', '#'))

# Extract columns
x = data[:, 0]  # First column (typically time or step)
y = data[:, 1]  # Second column (typically the value you're plotting)

# Plot the data
plt.plot(x, y)
plt.xlabel('Residue')  # Label for x-axis
plt.ylabel('RMSF')            # Label for y-axis
plt.title('RMSF protein')  # Title of the plot

# Show the plot
plt.savefig('6g99_ZnF_UGGUG_protein_rmsf.png', dpi=100)
plt.show()

<a id="sec3"></a>
## 3. Protein–RNA distance map (MDAnalysis)

Builds a static distance matrix between protein Cα atoms and RNA C1' atoms using MDAnalysis, then renders it as a heatmap. Applied to the FUS RRM–RNA complex.

<sub>Source script: <code>Contact_map_MDA.ipynb</code></sub>

In [ ]:
import MDAnalysis as mda
from MDAnalysis.tests.datafiles import PDB_small
from MDAnalysis.analysis import distances

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
u = mda.Universe('6gbm_RRM_whole_md_stripped.gro', '6gbm_RRM_whole_md_strip+fit.xtc')
print (u)
print(u.residues)

In [ ]:
print(u.residues.names)

In [ ]:
print (u.select_atoms("protein"))

In [ ]:
protein = u.select_atoms("protein")

In [ ]:
print(u.select_atoms("nucleic"))

In [ ]:
RNA= u.select_atoms("nucleic")

In [ ]:
prot_ca = protein.select_atoms("name CA")
print(prot_ca)

In [ ]:
RNA_c1 = RNA.select_atoms("name C1'")

In [ ]:
print (RNA.names)

In [ ]:
print (RNA_c1)

In [ ]:
res_dist = distances.distance_array(prot_ca, RNA_c1,
                                    box=u.dimensions)

In [ ]:
fig2, ax2 = plt.subplots()
im2 = ax2.imshow(res_dist, origin='upper')

prot_com = protein.center_of_mass(compound='residues')
RNA_com = RNA.center_of_mass(compound='residues')

n_prot = len(prot_com)
n_RNA = len(RNA_com)


# add residue ID labels to axes
tick_interval = 10
ax2.set_yticks(np.arange(n_prot)[::tick_interval])
ax2.set_xticks(np.arange(n_RNA)[::tick_interval])
ax2.set_yticklabels(protein.residues.resids[::tick_interval])
ax2.set_xticklabels(RNA.residues.resids[::tick_interval])

# add figure labels and titles
plt.ylabel('protein')
plt.xlabel('RNA')
plt.title('Distance between center-of-mass')

# colorbar
cbar2 = fig2.colorbar(im2)
cbar2.ax.set_ylabel('Distance (Angstrom)')


<a id="sec4"></a>
## 4. Contact counting over trajectory

Counts protein–RNA atom-pair contacts within a distance cutoff across all trajectory frames using MDAnalysis, producing a time-series of contact number.

<sub>Source script: <code>Contact_numbers.ipynb</code></sub>

In [ ]:
import MDAnalysis as mda
from MDAnalysis.tests.datafiles import PSF, DCD
from MDAnalysis.analysis import contacts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
u = mda.Universe('6gbm_RRM_whole_md_stripped.tpr', '6gbm_RRM_whole_md_strip+fit.xtc')
print (u)
print(u.residues)

In [ ]:
sel_prot = "protein"
sel_RNA = "nucleic"
protein = u.select_atoms(sel_prot)
RNA = u.select_atoms(sel_RNA)

In [ ]:
def contacts_within_cutoff(u, protein, RNA, radius=3.5):
    timeseries = []
    for ts in u.trajectory:
        # calculate distances between group_a and group_b
        dist = contacts.distance_array(protein.positions, RNA.positions)
        # determine which distances <= radius
        n_contacts = contacts.contact_matrix(dist, radius).sum()
        timeseries.append([ts.frame, n_contacts])
    return np.array(timeseries)


In [ ]:
ca = contacts_within_cutoff(u, protein, RNA, radius=3.5)
ca.shape

ca_df = pd.DataFrame(ca, columns=['Frame',
                                  '# Contacts'])
ca_df.head()

In [ ]:


ca_df.plot(x='Frame')
plt.ylabel('# contacts 3.5 A')

<a id="sec5"></a>
## 5. Residue-level contact frequency heatmap

Calculates pairwise residue-level protein–RNA distances across the full trajectory, converts to a contact-frequency matrix (fraction of frames within cutoff), and plots an annotated heatmap. Applied to the FUS RRM–RNA system.

<sub>Source script: <code>H-bonds.ipynb</code></sub>

In [ ]:
import pickle
import numpy as np
np.set_printoptions(linewidth=100)
import pandas as pd

import matplotlib.pyplot as plt

import MDAnalysis as mda
from MDAnalysis.tests.datafiles import waterPSF, waterDCD
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis

# the next line is necessary to display plots in Jupyter


In [ ]:
u = mda.Universe('6snj_RRM_whole_md_stripped.gro', '6snj_RRM_whole_md_strip+fit.xtc')
print (u)
print(u.residues)

In [ ]:
from MDAnalysis.analysis import distances
from MDAnalysis.analysis.distances import distance_array, dist

# Define selection for protein and RNA residues
protein = u.select_atoms('protein')  # Selects all protein atoms
rna = u.select_atoms('nucleic')  # Selects RNA residues (A, U, G, C for RNA bases)

# Define cutoff distance (in Angstroms)
cutoff = 3.0  # Typical contact distance for protein-RNA interactions

# Initialize an empty matrix to store contact counts
contact_matrix = np.zeros((len(protein.residues), len(rna.residues)))

# Total number of frames in the trajectory
total_frames = len(u.trajectory)

# Loop through all frames of the trajectory
for ts in u.trajectory:
    # Calculate pairwise distances between protein and RNA atoms
    protein_coords = protein.positions
    rna_coords = rna.positions

    # Use MDAnalysis to calculate the distances between protein and RNA atoms
    dist_matrix = distance_array(protein_coords, rna_coords)

    # Find contacts (distances below the cutoff)
    contact_mask = dist_matrix < cutoff  # Boolean array: True for contacts

    # Count contacts for each protein residue vs RNA residue
    for i, protein_res in enumerate(protein.residues):
        for j, rna_res in enumerate(rna.residues):
            # Get the indices of atoms in each residue
            protein_res_atoms = protein_res.atoms
            rna_res_atoms = rna_res.atoms
            
            # Get the indices of the atoms in the contact_mask matrix
            protein_indices = protein_res_atoms.indices
            rna_indices = rna_res_atoms.indices
            
            # Map the global atom indices to the local indices for the contact_mask
            protein_local_indices = np.where(np.isin(protein.indices, protein_indices))[0]
            rna_local_indices = np.where(np.isin(rna.indices, rna_indices))[0]
            
            # Get the submatrix of the contact_mask for these atoms using local indices
            atom_contacts = contact_mask[protein_local_indices[:, None], rna_local_indices]

            # If there is at least one contact (True), increment the contact count for this residue pair
            if np.any(atom_contacts):
                contact_matrix[i, j] += 1


# Calculate the percentage of frames with at least one contact for each residue pair
contact_percentage = (contact_matrix / total_frames) * 100

# Plotting the heatmap
plt.figure(figsize=(12, 10))
plt.imshow(contact_percentage, cmap='coolwarm', interpolation='nearest')
plt.colorbar(label='Percentage of Frames with Contact')

# Set the axis labels to residue names
# Use residue names with their indices if IDs are not available
protein_labels = [residue.resname + str(i + 1) for i, residue in enumerate(protein.residues)]
rna_labels = [residue.resname + str(i + 1) for i, residue in enumerate(rna.residues)]

plt.xticks(np.arange(len(rna.residues)), rna_labels, rotation=90)
plt.yticks(np.arange(len(protein.residues)), protein_labels)

# Add titles and labels
plt.title('Percentage of Frames with Protein-RNA Residue Contacts')
plt.xlabel('RNA Residue')
plt.ylabel('Protein Residue')

# Save the plot as an image (e.g., PNG)
plt.tight_layout()
plt.savefig('protein_rna_contact_percentage_heatmap.png', dpi=300)  # Save as PNG with high resolution

# Optionally, you can also save in other formats:
# plt.savefig('protein_rna_contact_percentage_heatmap.pdf')  # Save as PDF

# Show the plot
plt.show()